<a href="https://colab.research.google.com/github/betulbilhan2/ai-carbon-tracker/blob/main/NB03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Gerekli kütüphaneleri kuralım
!pip install -q pytorch-tabnet scikit-learn pandas numpy pyarrow

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from pytorch_tabnet.tab_model import TabNetRegressor
import torch

base_path = '/content/drive/MyDrive/terkentech'
print("Ortam hazır ve Drive bağlandı!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.2 MB/s eta 0:00:00
Mounted at /content/drive
Ortam hazır ve Drive bağlandı!


In [ ]:
# Notebook 2'den gelen ölçeklenmiş train/val/test setlerini eksiksiz okuyalım
train_df = pd.read_parquet(f'{base_path}/01_processed/tabnet_train_scaled.parquet')
val_df = pd.read_parquet(f'{base_path}/01_processed/tabnet_val_scaled.parquet')
test_df = pd.read_parquet(f'{base_path}/01_processed/tabnet_test_scaled.parquet')

# Metadata'dan hedef sütun ve özellik listelerini çekelim
with open(f'{base_path}/01_processed/feature_metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

target_col = metadata.get("target_column", "CarbonEmission")
features = metadata.get("model_features", [c for c in train_df.columns if c != target_col])

numerical_cols = metadata.get("numerical_columns", [])
categorical_cols = metadata.get("categorical_columns", [])

# X ve y setlerini oluşturalım
X_train = train_df[features]
y_train = train_df[target_col].values

X_val = val_df[features]
y_val = val_df[target_col].values

X_test = test_df[features]
y_test = test_df[target_col].values

print(f"Eğitim seti boyutu: {X_train.shape}")
print(f"Doğrulama seti boyutu: {X_val.shape}")
print(f"Test seti boyutu: {X_test.shape}")

Eğitim seti boyutu: (7000, 17)
Doğrulama seti boyutu: (1500, 17)
Test seti boyutu: (1500, 17)


In [ ]:
# Sabit tahmin baseline modeli (Ortalama)
mean_pred = np.mean(y_train)

test_preds_mean = np.full_like(y_test, mean_pred)

baseline_mae = mean_absolute_error(y_test, test_preds_mean)
baseline_rmse = np.sqrt(mean_squared_error(y_test, test_preds_mean))
baseline_r2 = r2_score(y_test, test_preds_mean)

print(f"Baseline (Mean) Test MAE: {baseline_mae:.2f}")
print(f"Baseline (Mean) Test RMSE: {baseline_rmse:.2f}")
print(f"Baseline (Mean) Test R²: {baseline_r2:.4f}")

Baseline (Mean) Test MAE: 778.02
Baseline (Mean) Test RMSE: 1026.96
Baseline (Mean) Test R²: -0.0003


In [ ]:
# Random Forest Modeli Eğitimi
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Test seti tahminleri
rf_preds = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

print(f"Random Forest Test MAE: {rf_mae:.2f}")
print(f"Random Forest Test RMSE: {rf_rmse:.2f}")
print(f"Random Forest Test R²: {rf_r2:.4f}")

Random Forest Test MAE: 303.79
Random Forest Test RMSE: 403.87
Random Forest Test R²: 0.8453


In [ ]:
# TabNet için kategorik indeksleri ve embedding boyutlarını hazırlayalım
cat_idxs = [X_train.columns.get_loc(col) for col in categorical_cols if col in X_train.columns]
cat_dims = [metadata["categorical_dims"][col] for col in categorical_cols if col in categorical_cols]
cat_emb_dim = [min(50, (x + 1) // 2) for x in cat_dims]

# TabNet Regressor tanımlama
tabnet_model = TabNetRegressor(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=cat_emb_dim,
    optimizer_params=dict(lr=2e-2),
    seed=42
)

# Eğitimi başlatma
tabnet_model.fit(
    X_train=X_train.values,
    y_train=y_train.reshape(-1, 1),
    eval_set=[(X_val.values, y_val.reshape(-1, 1))],
    eval_name=['val'],
    eval_metric=['mae'],
    max_epochs=25,
    patience=5,
    batch_size=256,
    virtual_batch_size=64,
    num_workers=0,
    drop_last=False
)

# Test seti tahminleri
tabnet_preds = tabnet_model.predict(X_test.values).flatten()

tabnet_mae = mean_absolute_error(y_test, tabnet_preds)
tabnet_rmse = np.sqrt(mean_squared_error(y_test, tabnet_preds))
tabnet_r2 = r2_score(y_test, tabnet_preds)

print(f"\nTabNet Test MAE: {tabnet_mae:.2f}")
print(f"TabNet Test RMSE: {tabnet_rmse:.2f}")
print(f"TabNet Test R²: {tabnet_r2:.4f}")

/usr/local/lib/python3.13/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 6125513.81086| val_mae: 2271.49316|  0:00:01s
epoch 1  | loss: 6029798.30171| val_mae: 2241.97656|  0:00:01s
epoch 2  | loss: 5825724.19371| val_mae: 2192.7229|  0:00:02s
epoch 3  | loss: 5500978.37486| val_mae: 2121.21533|  0:00:03s
epoch 4  | loss: 5084336.32571| val_mae: 2020.24316|  0:00:04s
epoch 5  | loss: 4589344.212| val_mae: 1862.79932|  0:00:05s
epoch 6  | loss: 4009712.75057| val_mae: 1714.52539|  0:00:06s
epoch 7  | loss: 3344780.99171| val_mae: 1561.75696|  0:00:06s
epoch 8  | loss: 2655687.33457| val_mae: 1437.73865|  0:00:07s
epoch 9  | loss: 1944321.74914| val_mae: 1018.37817|  0:00:08s
epoch 10 | loss: 1324291.80271| val_mae: 905.09424|  0:00:09s
epoch 11 | loss: 860920.41386| val_mae: 569.1582|  0:00:10s
epoch 12 | loss: 538996.41157| val_mae: 464.61472|  0:00:12s
epoch 13 | loss: 342623.79754| val_mae: 456.42426|  0:00:13s
epoch 14 | loss: 232309.86964| val_mae: 335.06354|  0:00:14s
epoch 15 | loss: 183286.98584| val_mae: 312.89304|  0:00:15s
epoch 1

/usr/local/lib/python3.13/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



TabNet Test MAE: 259.21
TabNet Test RMSE: 357.68
TabNet Test R²: 0.8787


In [ ]:
# Karşılaştırma Tablosu ve Model Seçim Mantığı
metrics_summary = {
    "mean_baseline": {"MAE": float(baseline_mae), "RMSE": float(baseline_rmse), "R2": float(baseline_r2)},
    "random_forest": {"MAE": float(rf_mae), "RMSE": float(rf_rmse), "R2": float(rf_r2)},
    "tabnet": {"MAE": float(tabnet_mae), "RMSE": float(tabnet_rmse), "R2": float(tabnet_r2)}
}

print("--- MODEL KARŞILAŞTIRMA ÖZETİ ---")
for model_name, metrics in metrics_summary.items():
    print(f"{model_name.upper()} -> MAE: {metrics['MAE']:.2f} | RMSE: {metrics['RMSE']:.2f} | R²: {metrics['R2']:.4f}")

# Klasör yapısını oluşturalım
os.makedirs(f'{base_path}/02_models/tabnet', exist_ok=True)
os.makedirs(f'{base_path}/02_models/rf', exist_ok=True)

# Nihai kazanan TabNet olduğu için kaydediyoruz
chosen_model_name = "tabnet"
tabnet_model.save_model(f'{base_path}/02_models/tabnet/tabnet_model_v1')
joblib.dump(rf_model, f'{base_path}/02_models/rf/rf_model_v1.pkl')

print("\n[KARAR]: TabNet daha düşük MAE ve yüksek R² skoru ile kazandı; nihai model olarak kaydedildi!")

metrics_summary["chosen_model"] = chosen_model_name

# Metrikleri JSON olarak kaydedelim
with open(f'{base_path}/02_models/tabnet_metrics_v1.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=4)

print("tabnet_metrics_v1.json başarıyla kaydedildi. Notebook 3 resmi olarak tamamlandı! 🚀")

--- MODEL KARŞILAŞTIRMA ÖZETİ ---
MEAN_BASELINE -> MAE: 778.02 | RMSE: 1026.96 | R²: -0.0003
RANDOM_FOREST -> MAE: 303.79 | RMSE: 403.87 | R²: 0.8453
TABNET -> MAE: 259.21 | RMSE: 357.68 | R²: 0.8787
Successfully saved model at /content/drive/MyDrive/terkentech/02_models/tabnet/tabnet_model_v1.zip

[KARAR]: TabNet daha düşük MAE ve yüksek R² skoru ile kazandı; nihai model olarak kaydedildi!
tabnet_metrics_v1.json başarıyla kaydedildi. Notebook 3 resmi olarak tamamlandı! 🚀


In [ ]:
import shutil
import os

# 03_models klasörünü oluşturalım
os.makedirs(f'{base_path}/03_models/tabnet', exist_ok=True)

# 02_models altındaki tabnet modelini 03_models altına taşıyalım/kopyalayalım
shutil.copytree(f'{base_path}/02_models/tabnet', f'{base_path}/03_models/tabnet', dirs_exist_ok=True)

# Metrik JSON dosyasını da 03_models altına kopyalayalım
if os.path.exists(f'{base_path}/02_models/tabnet_metrics_v1.json'):
    shutil.copy(f'{base_path}/02_models/tabnet_metrics_v1.json', f'{base_path}/03_models/tabnet_metrics_v1.json')

print("Modeller ve metrikler başarıyla '03_models' klasörüne aktarıldı! 🚀")

Modeller ve metrikler başarıyla '03_models' klasörüne aktarıldı! 🚀
